# Commutator Antisymmetry Verification: [A, B] = -[B, A]

This notebook verifies that the commutator expressions computed by `qcombo` satisfy the antisymmetry relation:

$$[A^{(M)}, B^{(N)}]_K = -[B^{(N)}, A^{(M)}]_K$$

where $M$ and $N$ denote the body numbers of operators $A$ and $B$, and $K$ denotes the body number of the commutator result.

Users can freely adjust the following parameters:
- `left_body`: body number of the left operator
- `right_body`: body number of the right operator
- `contraction_body`: body number of the commutator result (single value or list)

In [1]:
import qcombo
from sympy import IndexedBase, symbols, preorder_traversal, expand, simplify
from sympy.tensor.indexed import Indexed
from IPython.display import display, Latex, Markdown

print(f"qcombo version: {qcombo.__version__}")

qcombo version: 0.2.0


In [2]:
def jupyterDisplay(expr, label=""):
    """Display a SymPy expression in LaTeX form within Jupyter"""
    latex_expr = qcombo.texExp(expr)
    if label:
        display(Markdown(f"**{label}:**"))
    display(Latex(f"$${latex_expr}$$"))


## Parameter Settings

Modify the following parameters to control the body numbers of the commutator:

In [3]:
# ============================================================
# Adjustable parameters: modify these values to verify different commutators
# ============================================================

# Body number of left operator A (1B, 2B, 3B, ...)
left_body = 2

# Body number of right operator B (1B, 2B, 3B, ...)
right_body = 2

# Target contraction body of the commutator result
# Can be a single integer or a list, e.g. [0, 1]
# Set to None to compute all possible body numbers
contraction_body = None  # None means all possible body numbers

# Whether to show detailed process
show_process = False

# Whether to use parallel computation
parallel = False

# ============================================================

# print(f"Left operator body: {left_body}B")
# print(f"Right operator body: {right_body}B")
# print(f"Target contraction body: {'all' if contraction_body is None else contraction_body}B")

## Compute [A, B]

Use `qcombo.easyCombo` to compute the commutator $[A^{(M)}, B^{(N)}]$:

In [4]:
# Compute [A, B]
print(f"Computing [{left_body}B, {right_body}B]...")
result_AB = qcombo.easyCombo(
    left=left_body, 
    right=right_body, 
    contraction=contraction_body,
    show_process=show_process,
    parallel=parallel,
    savefile=False
)

expr_dict_AB = result_AB.expr_dict

# print(f"\n[A, B] result contains {len(expr_dict_AB)} non-zero terms:")
# for key in expr_dict_AB.keys():
#     print(f"  - {key}")

Computing [2B, 2B]...


## Compute [B, A]

Use the same parameters but swap the order of left and right operators:

In [5]:
# Compute [B, A] — swap left and right operators
print(f"Computing [{right_body}B, {left_body}B]...")
result_BA = qcombo.easyCombo(
    left=right_body, 
    right=left_body, 
    contraction=contraction_body,
    show_process=show_process,
    parallel=parallel,
    savefile=False
)

expr_dict_BA = result_BA.expr_dict

# print(f"\n[B, A] result contains {len(expr_dict_BA)} non-zero terms:")
# for key in expr_dict_BA.keys():
#     print(f"  - {key}")

Computing [2B, 2B]...


## Verify Antisymmetry: [A, B] + [B, A] = 0

For each contraction body, verify that $[A, B]_K + [B, A]_K = 0$:

In [6]:
from sympy import IndexedBase, preorder_traversal
from sympy.tensor.indexed import Indexed


def swap_operator_bases(expr):
    """
    Swap the G and H operator bases in the expression.
    Since easyCombo always uses G for the left operator and H for the right operator,
    when computing [B, A], G corresponds to B and H corresponds to A.
    To compare with [A, B], we need to swap G↔H in [B, A].
    """
    G = IndexedBase('G')
    H = IndexedBase('H')
    # Use a temporary symbol to avoid conflicts
    tmp = IndexedBase('__TMP__')
    return expr.xreplace({G: tmp, H: G}).xreplace({tmp: H})

def reIndices(expr):
    """
    Re-index the expression.
    e.g. A[i,j]*B[k,l] -> A[a,b]*B[c,d]
    """
    canon_expr = qcombo.canonical.canonicalize(expr.expand(),parallel=False,show_process=False)
    reIndices_expr,indices_set =qcombo.tools.indicesMultToSimp(canon_expr,parallel=False,show_process=False)

    # For multi-body expressions, we also need to restore index antisymmetry
    re_AntiSymetry_expr = qcombo.tools.antisymmetrize_expr(reIndices_expr)

    simplified_expr = qcombo.simplifyUseBoth(re_AntiSymetry_expr.expand(),show_process=False,parallel=False)
    united_expr = qcombo.MergeSameMatrixElement(simplified_expr)
    return united_expr


def verify_antisymmetry(expr_dict_AB, expr_dict_BA):
    """
    Verify antisymmetry: [A, B] = -[B, A]
    
    For each contraction body, check whether sum = [A,B]_K + [B,A]_K equals 0.
    Note: G/H in [B,A]_K need to be swapped before comparison.
    """
    all_passed = True
    results = []
    
    # Get keys for all body numbers
    all_keys = set(expr_dict_AB.keys()) | set(expr_dict_BA.keys())
    
    for key in sorted(all_keys):
        expr_AB = expr_dict_AB.get(key, 0)
        expr_BA_raw = expr_dict_BA.get(key, 0)

        # [A,B] 
        
        # Swap G↔H in [B,A] to make it comparable with [A,B]
        if expr_BA_raw != 0:
            expr_BA = swap_operator_bases(expr_BA_raw)
        else:
            expr_BA = 0

        # After swapping, re-index to avoid situations where
        # G^a_b H^b_a != G^b_a H^a_b, even though they are essentially the same
        expr_AB = reIndices(expr_AB)
        expr_BA = reIndices(expr_BA)

        
        # Compute [A,B] + [B,A]
        total = expand(expr_AB + expr_BA)


        is_zero = (total == 0)
        
        results.append({
            'key': key,
            'expr_AB': expr_AB,
            'expr_BA_swapped': expr_BA,
            'sum': total,
            'is_zero': is_zero
        })
        
        if not is_zero:
            all_passed = False
    
    return all_passed, results


# Execute verification
all_passed, verification_results = verify_antisymmetry(expr_dict_AB, expr_dict_BA)

# Display results
print("=" * 60)
print("Verification Results:")
print("=" * 60)

for res in verification_results:
    key = res['key']
    status = "✅ Passed" if res['is_zero'] else "❌ Failed"
    print(f"\n{key}: {status}")
    if res['expr_AB'] != 0:
        print(f"  [A,B]_{key[-2:]} non-zero terms: {len(list(res['expr_AB'].args)) if hasattr(res['expr_AB'], 'args') and res['expr_AB'] != 0 else 0}")
    if not res['is_zero']:
        print(f"  [A,B] + [B,A] = {res['sum']}")

print("\n" + "=" * 60)
if all_passed:
    print("🎉 All body antisymmetry verification passed! [A, B] = -[B, A] holds.")
else:
    print("⚠️ Some body antisymmetry verifications failed, please check the output above.")
print("=" * 60)

Verification Results:

0B_lambda1B: ✅ Passed
  [A,B]_1B non-zero terms: 4

0B_lambda2B: ✅ Passed
  [A,B]_2B non-zero terms: 3

0B_lambda3B: ✅ Passed
  [A,B]_3B non-zero terms: 3

1B_lambda1B: ✅ Passed
  [A,B]_1B non-zero terms: 3

1B_lambda2B: ✅ Passed
  [A,B]_2B non-zero terms: 4

2B_lambda1B: ✅ Passed
  [A,B]_1B non-zero terms: 3

3B_lambda1B: ✅ Passed
  [A,B]_1B non-zero terms: 3

🎉 All body antisymmetry verification passed! [A, B] = -[B, A] holds.


## Detailed Display of Commutator Expressions for Each Body

The following cells show the explicit forms of $[A,B]_K$ and $[B,A]_K$ (after swapping G/H):

In [7]:
for res in verification_results:
    key = res['key']
    display(Markdown(f"### {key}"))
    
    if res['expr_AB'] != 0:
        jupyterDisplay(res['expr_AB'], label="[A, B]")
    else:
        display(Markdown("[A, B] = 0"))
    
    if res['expr_BA_swapped'] != 0:
        jupyterDisplay(res['expr_BA_swapped'], label="[B, A] (after swapping G↔H)")
    else:
        display(Markdown("[B, A] (after swapping) = 0"))
    
    if res['is_zero']:
        display(Markdown(f"**[A,B] + [B,A] = 0  ✅ Passed**"))
    else:
        jupyterDisplay(res['sum'], label="[A,B] + [B,A] (non-zero!)")
    
    display(Markdown("---"))

### 0B_lambda1B

**[A, B]:**

<IPython.core.display.Latex object>

**[B, A] (after swapping G↔H):**

<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

**[B, A] (after swapping G↔H):**

<IPython.core.display.Latex object>

**[A,B] + [B,A] = 0  ✅ Passed**

---

## Batch Testing: Iterate Over Multiple Body Number Combinations

The following cell automatically tests the antisymmetry for multiple $(M, N)$ combinations:

In [8]:
import itertools

# ============================================================
# Batch test parameters
# ============================================================
# List of left body numbers to test
left_body_list = [1, 2]

# List of right body numbers to test
right_body_list = [1, 2]

# Whether to show only the summary
summary_only = True
# ============================================================

batch_results = []

for M, N in itertools.product(left_body_list, right_body_list):
    print(f"\nTesting [{M}B, {N}B] vs [{N}B, {M}B] ...")
    
    # Compute [A, B]
    r_AB = qcombo.easyCombo(
        left=M, right=N, contraction=None,
        show_process=False, parallel=False, savefile=False
    )
    
    # Compute [B, A]
    r_BA = qcombo.easyCombo(
        left=N, right=M, contraction=None,
        show_process=False, parallel=False, savefile=False
    )
    
    passed, results = verify_antisymmetry(r_AB.expr_dict, r_BA.expr_dict)
    
    if not summary_only:
        for res in results:
            status = "✅" if res['is_zero'] else "❌"
            print(f"  {res['key']}: {status}")
    
    status_str = "✅ Passed" if passed else "❌ Failed"
    print(f"  [{M}B, {N}B] overall: {status_str}")
    batch_results.append(((M, N), passed, results))

# Summary
print("\n" + "=" * 60)
print("Batch Test Summary:")
print("=" * 60)
all_batch_passed = all(p for _, p, _ in batch_results)
for (M, N), passed, _ in batch_results:
    status = "✅" if passed else "❌"
    print(f"  [{M}B, {N}B]: {status}")

if all_batch_passed:
    print(f"\n🎉 All {len(batch_results)} combinations passed the antisymmetry verification!")
else:
    failed = [(M, N) for (M, N), p, _ in batch_results if not p]
    print(f"\n⚠️ {len(failed)} combinations failed: {failed}")


Testing [1B, 1B] vs [1B, 1B] ...
  [1B, 1B] overall: ✅ Passed

Testing [1B, 2B] vs [2B, 1B] ...
  [1B, 2B] overall: ✅ Passed

Testing [2B, 1B] vs [1B, 2B] ...
  [2B, 1B] overall: ✅ Passed

Testing [2B, 2B] vs [2B, 2B] ...
  [2B, 2B] overall: ✅ Passed

Batch Test Summary:
  [1B, 1B]: ✅
  [1B, 2B]: ✅
  [2B, 1B]: ✅
  [2B, 2B]: ✅

🎉 All 4 combinations passed the antisymmetry verification!


## Manual Verification: 1B+1B → 0B Example

Below is a manual verification of the antisymmetry for the 1-body commutator, to help understand the verification logic:

In [9]:
from sympy import IndexedBase, symbols, expand, preorder_traversal
from sympy.tensor.indexed import Indexed

# Compute 1B commutator
print("Computing [1B, 1B] ...")
r_1B1B = qcombo.easyCombo(1, 1, 0,show_process=False, savefile=False, parallel=False)

# View results for all body numbers
for key, expr in r_1B1B.expr_dict.items():
    print(f"\n{key}:")
    jupyterDisplay(expr)

# Manually check the 0B part:
# [A,B]_0B = sum_{ab} (n_a - n_b) A^a_b B^b_a
# [B,A]_0B = sum_{ab} (n_a - n_b) B^a_b A^b_a = -sum_{ab} (n_a - n_b) A^b_a B^a_b
# Swap dummy indices a<->b: = -sum_{ab} (n_b - n_a) A^a_b B^b_a = -[A,B]_0B
print("\n" + "=" * 60)
print("Manual Verification Logic:")
print("[A,B]_0B = Σ_{ab} (n_a - n_b) A^a_b B^b_a")
print("[B,A]_0B = Σ_{ab} (n_a - n_b) B^a_b A^b_a = Σ_{ba} (n_b - n_a) A^a_b B^b_a")
print("Swap dummy indices a↔b: = Σ_{ab} (n_b - n_a) A^a_b B^b_a = -[A,B]_0B  ✓")
print("=" * 60)

Computing [1B, 1B] ...

0B_lambda1B:


<IPython.core.display.Latex object>


Manual Verification Logic:
[A,B]_0B = Σ_{ab} (n_a - n_b) A^a_b B^b_a
[B,A]_0B = Σ_{ab} (n_a - n_b) B^a_b A^b_a = Σ_{ba} (n_b - n_a) A^a_b B^b_a
Swap dummy indices a↔b: = Σ_{ab} (n_b - n_a) A^a_b B^b_a = -[A,B]_0B  ✓


Note that in the expression above, since the summation indices ab are dummy indices, they can be interchanged. Therefore, after swapping the matrix element symbols, the resulting commutator [B,A] still requires further manipulation of the dummy indices before one can explicitly verify that [A,B] = -[B,A].

## Summary

This notebook verifies the fundamental antisymmetry property of commutators:

$$[A^{(M)}, B^{(N)}]_K = -[B^{(N)}, A^{(M)}]_K$$

- **Parameterized design**: freely adjust `left_body`, `right_body`, and `contraction_body` to control operator body numbers and target contraction bodies
- **Automated verification**: automatically computes $[A,B]$ and $[B,A]$, then compares whether they are opposites after swapping the G↔H bases
- **Batch testing**: supports testing multiple $(M,N)$ combinations at once